# CC3104 – Aprendizaje por Refuerzo
## Laboratorio 1 — Task 4 — Dictamen técnico (Entrega Final)

**Integrantes:**
- Sergio Orellana — 221122
- Rodrigo Mansilla — 22611
- Ricardo Chuy — 221007

---


### Instrucciones

Con base en los resultados de la implementación del Task 3, redacten un dictamen técnico dirigido a la gerencia de la empresa de logística.

---
### 1. Comparación cuantitativa de las tres políticas

Comparen cuantitativamente las tres políticas evaluadas. ¿Cuál produce los valores más altos? ¿En qué estados difieren más? ¿La política greedy derivada de la política aleatoria es mejor que la política determinista que diseñaron manualmente?

In [1]:
# Codigo de apoyo: reconstruye el MDP y el evaluador del Task 3 para
# generar las metricas cuantitativas usadas en la respuesta de la pregunta 1.
from collections import defaultdict

import numpy as np
import pandas as pd


class DroneGridWorldMDP:
    """MDP tabular para una mision de entrega y regreso en una cuadricula 5x5."""

    def __init__(self, grid_size=5, base=(2, 2), delivery=(0, 4), max_battery=10):
        self.grid_size = grid_size
        self.base = base
        self.delivery = delivery
        self.max_battery = max_battery

        self.phases = ("en_camino", "regresando")
        self.actions = ("Norte", "Sur", "Este", "Oeste", "Esperar")

        self.terminal_success = ("COMPLETO",)
        self.terminal_failure = ("FALLA",)
        self.terminal_states = {self.terminal_success, self.terminal_failure}

        self.states = [
            (row, col, phase, battery)
            for row in range(grid_size)
            for col in range(grid_size)
            for phase in self.phases
            for battery in range(1, max_battery + 1)
        ]
        self.states.extend([self.terminal_success, self.terminal_failure])

        self.action_delta = {
            "Norte": (-1, 0), "Sur": (1, 0), "Este": (0, 1), "Oeste": (0, -1), "Esperar": (0, 0),
        }
        self.left_of = {"Norte": "Oeste", "Sur": "Este", "Este": "Norte", "Oeste": "Sur"}
        self.right_of = {"Norte": "Este", "Sur": "Oeste", "Este": "Sur", "Oeste": "Norte"}

    def is_terminal(self, state):
        return state in self.terminal_states

    def action_outcomes(self, action):
        if action == "Esperar":
            return [("Esperar", 1.0)]
        return [(action, 0.8), (self.left_of[action], 0.1), (self.right_of[action], 0.1)]

    def _apply_realized_action(self, state, realized_action):
        row, col, phase, battery = state
        delta_row, delta_col = self.action_delta[realized_action]
        candidate = (row + delta_row, col + delta_col)
        valid_move = 0 <= candidate[0] < self.grid_size and 0 <= candidate[1] < self.grid_size

        if realized_action == "Esperar":
            next_position, battery_cost = (row, col), 1
        elif valid_move:
            next_position, battery_cost = candidate, 1
        else:
            next_position, battery_cost = (row, col), 0

        next_battery = battery - battery_cost

        if phase == "regresando" and next_position == self.base:
            return self.terminal_success, 50.0
        if next_battery <= 0:
            return self.terminal_failure, -100.0
        if phase == "en_camino" and next_position == self.delivery:
            return (next_position[0], next_position[1], "regresando", next_battery), 100.0

        return (next_position[0], next_position[1], phase, next_battery), -1.0

    def transitions(self, state, action):
        if self.is_terminal(state):
            return [(1.0, state, 0.0)]
        aggregated = defaultdict(float)
        for realized_action, probability in self.action_outcomes(action):
            next_state, reward = self._apply_realized_action(state, realized_action)
            aggregated[(next_state, reward)] += probability
        return [(p, s, r) for (s, r), p in aggregated.items()]


def action_value(mdp, state, action, values, gamma):
    return sum(p * (r + gamma * values[s]) for p, s, r in mdp.transitions(state, action))


def policy_evaluation(mdp, policy, gamma=0.95, theta=1e-8, max_iterations=10_000):
    values = {state: 0.0 for state in mdp.states}
    for iteration in range(1, max_iterations + 1):
        new_values = values.copy()
        delta = 0.0
        for state in mdp.states:
            if mdp.is_terminal(state):
                new_values[state] = 0.0
                continue
            updated_value = sum(
                action_probability * action_value(mdp, state, action, values, gamma)
                for action, action_probability in policy[state].items()
            )
            new_values[state] = updated_value
            delta = max(delta, abs(updated_value - values[state]))
        values = new_values
        if delta < theta:
            return values, iteration, delta
    raise RuntimeError("La evaluacion no convergio dentro del limite configurado.")


def make_uniform_random_policy(mdp):
    probability = 1.0 / len(mdp.actions)
    return {s: {a: probability for a in mdp.actions} for s in mdp.states if not mdp.is_terminal(s)}


def manual_action(mdp, state):
    row, col, phase, _ = state
    target_row, target_col = mdp.delivery if phase == "en_camino" else mdp.base
    if row > target_row:
        return "Norte"
    if row < target_row:
        return "Sur"
    if col < target_col:
        return "Este"
    if col > target_col:
        return "Oeste"
    return "Esperar"


def make_manual_policy(mdp):
    return {s: {manual_action(mdp, s): 1.0} for s in mdp.states if not mdp.is_terminal(s)}


def make_greedy_policy_from_values(mdp, values, gamma=0.95):
    policy = {}
    for state in mdp.states:
        if mdp.is_terminal(state):
            continue
        q = {a: action_value(mdp, state, a, values, gamma) for a in mdp.actions}
        best_value = max(q.values())
        best_action = next(a for a in mdp.actions if np.isclose(q[a], best_value))
        policy[state] = {best_action: 1.0}
    return policy


mdp = DroneGridWorldMDP()
GAMMA = 0.95

random_policy = make_uniform_random_policy(mdp)
manual_policy = make_manual_policy(mdp)

random_values, random_it, _ = policy_evaluation(mdp, random_policy, gamma=GAMMA)
manual_values, manual_it, _ = policy_evaluation(mdp, manual_policy, gamma=GAMMA)
greedy_policy = make_greedy_policy_from_values(mdp, random_values, gamma=GAMMA)
greedy_values, greedy_it, _ = policy_evaluation(mdp, greedy_policy, gamma=GAMMA)

initial_state = (mdp.base[0], mdp.base[1], "en_camino", mdp.max_battery)
non_terminal_states = [s for s in mdp.states if not mdp.is_terminal(s)]

print("V(estado inicial) [base, en_camino, bateria=10]")
print(f"  aleatoria = {random_values[initial_state]:.2f}")
print(f"  determinista manual = {manual_values[initial_state]:.2f}")
print(f"  greedy (derivada de aleatoria) = {greedy_values[initial_state]:.2f}")

mr = np.array([manual_values[s] - random_values[s] for s in non_terminal_states])
mg = np.array([manual_values[s] - greedy_values[s] for s in non_terminal_states])

print(f"\nmanual - aleatoria: media={mr.mean():.2f}  max={mr.max():.2f}  min={mr.min():.2f}")
print(f"manual - greedy:   media={mg.mean():.2f}  max={mg.max():.2f}  min={mg.min():.2f}")
print(f"  manual > greedy en {np.mean(mg > 1e-6) * 100:.1f}% de los estados")
print(f"  greedy > manual en {np.mean(mg < -1e-6) * 100:.1f}% de los estados")
print(f"  empate en {np.mean(np.abs(mg) <= 1e-6) * 100:.1f}% de los estados")

diffs_mg = sorted(((manual_values[s] - greedy_values[s], s) for s in non_terminal_states), reverse=True)
print("\nEstados donde manual supera mas a greedy:")
for d, s in diffs_mg[:3]:
    print(f"  {s}: manual={manual_values[s]:.2f} greedy={greedy_values[s]:.2f} diff={d:+.2f}")
print("Estados donde greedy supera mas a manual:")
for d, s in diffs_mg[-3:]:
    print(f"  {s}: manual={manual_values[s]:.2f} greedy={greedy_values[s]:.2f} diff={d:+.2f}")


V(estado inicial) [base, en_camino, bateria=10]
  aleatoria = -62.08
  determinista manual = 79.02
  greedy (derivada de aleatoria) = 74.46

manual - aleatoria: media=63.26  max=160.79  min=-42.98
manual - greedy:   media=-7.55  max=47.72  min=-115.73
  manual > greedy en 29.8% de los estados
  greedy > manual en 63.8% de los estados
  empate en 6.4% de los estados

Estados donde manual supera mas a greedy:
  (2, 0, 'en_camino', 10): manual=32.69 greedy=-15.04 diff=+47.72
  (3, 1, 'en_camino', 10): manual=23.24 greedy=-19.02 diff=+42.26
  (1, 0, 'en_camino', 10): manual=43.19 greedy=1.88 diff=+41.31
Estados donde greedy supera mas a manual:
  (4, 0, 'en_camino', 6): manual=-80.21 greedy=-34.33 diff=-45.89
  (4, 0, 'en_camino', 7): manual=-76.93 greedy=-30.36 diff=-46.56
  (0, 4, 'en_camino', 1): manual=-100.00 greedy=15.73 diff=-115.73


**Respuesta:**

**Comparacion cuantitativa (gamma = 0.95, theta = 1e-8):**

| Politica | V(base, en_camino, bateria=10) | Iteraciones a convergencia |
|---|---|---|
| Uniforme aleatoria | -62.08 | 44 |
| Determinista fija (manual) | 79.02 | 25 |
| Greedy derivada de la aleatoria | 74.46 | 198 |

En el estado inicial tipico de una mision cuando el dron está en la base, en camino, bateria llena el orden es **manual > greedy > aleatoria**, con una diferencia enorme entre la politica aleatoria y las otras dos: la aleatoria produce un valor **negativo** , mientras que las dos politicas dirigidas producen valores positivos de +74 a +79. Esto confirma que sin una regla de decision dirigida al objetivo, el dron deambula, gasta bateria y con frecuencia falla la mision antes de completar la entrega y el regreso, el costo esperado de ese deambular domina sobre las pocas veces que llega por azar.

**¿Donde difieren mas las politicas?**

- **Aleatoria vs. manual/greedy:** la diferencia es mayor en los estados cercanos al punto de entrega con bateria alta , con diferencias de hasta 160 puntos frente a la aleatoria. Ahi el costo de oportunidad de moverse al azar en vez de dirigirse a la entrega es maximo, porque la entrega (+100) esta a pocos pasos y la aleatoria la desaprovecha sistematicamente.
- **Manual vs. greedy:** la diferencia mas grande **no** esta en los estados "normales" sino en los estados de **riesgo de bateria**, en las esquinas mas alejadas de la base y la entrega. Con bateria alta y cerca de la ruta directa, la manual gana por margenes moderados , porque su regla "vertical primero, horizontal despues" es casi optima. Pero en estados como en una esquina opuesta a la entrega o bateria critica, la greedy gana por margenes muy grandes , porque la politica manual **ignora la bateria por completo** en su regla de decision  y puede conducir al dron directo a una falla segura , mientras que la greedy, al derivarse de una funcion de valor que si incorpora el riesgo de quedarse sin bateria, elige rutas mas conservadoras en esos estados.

**¿Es la greedy derivada de la aleatoria mejor que la manual?**

Depende de la region del espacio de estados, y la respuesta agregada es reveladora, contando estado por estado, **la greedy supera a la manual en el 63.8% de los estados no terminales, empatan en 6.4% y la manual gana solo en 29.8%**. Sin embargo, en el estado inicial de una mision tipica la manual sigue siendo superior . La explicacion es que la manual es eficiente en el "camino feliz"  pero fragil en los bordes del espacio de estados donde la bateria es escasa, mientras que la greedy , al heredar la conciencia de riesgo de la politica aleatoria que evaluo primero, aunque esa politica en si sea mala, generaliza mejor a estados de riesgo que la manual nunca fue disenada para manejar. 

En terminos de gerencia, la politica manual es mas eficiente en el escenario promedio, pero la politica greedy es mas robusta ante escenarios de bateria baja, que son precisamente los que producen las fallas mas costosas operacionalmente.


---
### 2. Análisis de sensibilidad a 𝛾

Analicen la sensibilidad del resultado al valor de 𝛾 que propusieron en la Tarea 1. Repitan la evaluación con al menos dos valores adicionales de 𝛾 y discutan cómo cambian los valores y la política greedy resultante.

In [2]:
# Repetimos la evaluacion de politicas con dos valores adicionales de gamma
# (0.70 y 0.99) y los comparamos contra el gamma propuesto en el Task 1 (0.95).
gammas = [0.70, 0.95, 0.99]

summary_rows = []
greedy_policies_by_gamma = {}
greedy_values_by_gamma = {}

for gamma in gammas:
    r_values, r_it, _ = policy_evaluation(mdp, random_policy, gamma=gamma)
    m_values, m_it, _ = policy_evaluation(mdp, manual_policy, gamma=gamma)
    g_policy = make_greedy_policy_from_values(mdp, r_values, gamma=gamma)
    g_values, g_it, _ = policy_evaluation(mdp, g_policy, gamma=gamma)

    greedy_policies_by_gamma[gamma] = g_policy
    greedy_values_by_gamma[gamma] = g_values

    summary_rows.append({
        "gamma": gamma,
        "V_aleatoria(inicial)": r_values[initial_state],
        "V_manual(inicial)": m_values[initial_state],
        "V_greedy(inicial)": g_values[initial_state],
        "iter_aleatoria": r_it,
        "iter_manual": m_it,
        "iter_greedy": g_it,
    })

gamma_summary = pd.DataFrame(summary_rows)
display(gamma_summary.round(4))

base_gamma = 0.95
base_policy = greedy_policies_by_gamma[base_gamma]

print("\nCambios en la politica greedy respecto a gamma = 0.95:")
for gamma in gammas:
    if gamma == base_gamma:
        continue
    other_policy = greedy_policies_by_gamma[gamma]
    diffs = sum(
        1
        for s in non_terminal_states
        if next(iter(base_policy[s])) != next(iter(other_policy[s]))
    )
    pct = 100 * diffs / len(non_terminal_states)
    print(f"  gamma={gamma}: {diffs}/{len(non_terminal_states)} estados ({pct:.1f}%) cambian de accion greedy")


,gamma,V_aleatoria(inicial),V_manual(inicial),V_greedy(inicial),iter_aleatoria,iter_manual,iter_greedy
0,0.70,-5.1660,23.2509,21.7359,32,22,56
1,0.95,-62.0750,79.0226,74.4621,44,25,198
2,0.99,-92.5428,93.6475,93.9124,46,26,215



Cambios en la politica greedy respecto a gamma = 0.95:
  gamma=0.7: 70/500 estados (14.0%) cambian de accion greedy
  gamma=0.99: 179/500 estados (35.8%) cambian de accion greedy


**Respuesta:**


| gamma | V_aleatoria(inicial) | V_manual(inicial) | V_greedy(inicial) | iter. greedy |
|---|---|---|---|---|
| 0.70 | -5.17 | 23.25 | 21.74 | 56 |
| 0.95 | -62.08 | 79.02 | 74.46 | 198 |
| 0.99 | -92.54 | 93.65 | 93.91 | 215 |

**Como cambian los valores:**

- **Los valores absolutos crecen (en magnitud) con gamma en ambas direcciones.** Con gamma = 0.70 los valores estan comprimidos cerca de cero porque las recompensas terminales , que llegan varios pasos en el futuro, se descuentan agresivamente, a los 10 pasos , asi que casi no influyen sobre la decision actual y el valor del estado inicial queda dominado por el costo de paso -1 acumulado. Con gamma = 0.99, el descuento a 10 pasos es $(0.99)^{10}\approx 0.90$, por lo que el drone "ve" las recompensas terminales casi sin atenuar y los valores se acercan mas a la magnitud de esas recompensas .
- **El numero de iteraciones hasta convergencia crece con gamma** (198 en la greedy con gamma=0.95 vs 215 con gamma=0.99), porque el operador de Bellman contrae mas lentamente cuando gamma se acerca a 1 ,la tasa de contraccion es justamente gamma, asi que theta=1e-8 tarda mas en alcanzarse.
- **El orden relativo entre politicas se mantiene mayormente**, pero con gamma=0.70 y 0.95, manual > greedy en el estado inicial, con **gamma=0.99 esa relacion se invierte** (93.91 > 93.65): al valorar casi por igual el futuro lejano, la politica greedy —mas cautelosa con la bateria en los estados de riesgo, termina superando incluso en el estado inicial a la politica manual, que sigue ignorando la bateria en su regla fija.

**Como cambia la politica greedy resultante:**

Comparando la politica greedy optimabajo cada gamma contra la de gamma=0.95:

- Con **gamma=0.70**, la accion greedy cambia en el **14.0%** de los estados no terminales (70/500). Los cambios se concentran en estados cercanos a los bordes donde, al valorar poco el futuro, la politica prefiere acciones "seguras" a corto plazo en lugar de rutas que solo pagan varios pasos despues.
- Con **gamma=0.99**, el cambio es mucho mayor: **35.8%** de los estados (179/500) cambian de accion greedy respecto a gamma=0.95. Con casi nulo descuento, la politica esta dispuesta a tomar rutas mas largas pero mas seguras porque el costo de esos pasos extra  pesa poco frente a asegurar la recompensa terminal.

**Conclusion de sensibilidad:** Valores bajos producen un agente cortoplacista que puede ignorar el riesgo de quedarse sin bateria por priorizar terminar rapido, mientras que valores altos producen un agente mas prudente pero mas costoso de calcular  y que reordena una fraccion importante de las decisiones. El valor gamma=0.95 propuesto en la Tarea 1 se mantiene como un punto intermedio razonable, pero la magnitud de estos cambios  indica que gamma debe tratarse como un hiperparametro a validar empiricamente contra el comportamiento observado, no como una constante fija de diseño.


---
### 3. Limitaciones del modelo

Identifiquen las limitaciones del modelo que implementaron respecto al problema real. ¿Qué aspectos del dominio de logística urbana no puede capturar este MDP? ¿Qué extensiones serían necesarias para que el modelo sea operacionalmente útil?

**Respuesta:**

El modelo implementado es un MDP tabular, completamente observable y de un solo agente, definido sobre una cuadrícula estática de 5×5. Frente a un sistema real de logística urbana con drones, presenta varias limitaciones importantes.

**1. Modela un solo dron.**
El sistema considera un dron operando de forma aislada. En una operación real, varios drones comparten espacio aéreo, rutas y estaciones de carga. Por ello, el modelo no representa colisiones, congestión ni asignación de pedidos entre drones. Incluir las posiciones de toda la flota aumentaría el espacio de estados de forma combinatoria.

**2. Supone observabilidad perfecta.**
El dron conoce con exactitud su posición, nivel de batería y fase de la misión. En la práctica, el GPS, los sensores y las estimaciones de batería tienen ruido. Por tanto, el problema se aproxima más a un POMDP, donde el agente mantiene una estimación del estado en lugar de observarlo directamente.

**3. Simplifica demasiado la dinámica urbana.**
El entorno tiene una sola base, un único destino y probabilidades de movimiento constantes. No representa pedidos variables, clima dependiente de la zona y el tiempo, obstáculos, tráfico aéreo, zonas restringidas ni cambios en el consumo de batería según el peso, el viento o el desgaste del dron.

**4. La recompensa no incluye restricciones operativas.**
La función de recompensa optimiza una sola misión, pero omite ventanas de entrega, prioridades, costos de recarga y mantenimiento, riesgos para terceros y restricciones regulatorias. Estos factores pueden ser tan importantes como minimizar la distancia o evitar que el dron se quede sin batería.

**Extensiones necesarias para un uso operacional:**

* Incorporar múltiples drones mediante un enfoque multiagente o descentralizado.
* Usar un POMDP o un estimador de estado que combine GPS, IMU y sensores de batería.
* Sustituir la cuadrícula fija por un grafo de rutas con obstáculos, zonas restringidas y condiciones meteorológicas variables.
* Modelar múltiples pedidos, ventanas de entrega y decisiones de asignación.
* Ampliar la función de recompensa para incluir seguridad, regulación y costos operativos, y validar sus ponderaciones antes del despliegue.


---
### 4. Recomendación final

Concluyan con una recomendación concreta: ¿es el MDP que diseñaron una base suficiente para construir un sistema de RL real para esta empresa? ¿Qué pasos siguientes recomendarían antes de proceder con la implementación completa?

**Respuesta:**

**¿Es este MDP una base suficiente para un sistema de RL real?**

Como punto de partida conceptual, sí. El modelo representa los principales objetivos en conflicto, como eficiencia, consumo de batería y seguridad, mediante una función de recompensa. También incorpora transiciones estocásticas y permite evaluar políticas con resultados consistentes. Esto demuestra que el equipo comprende la dinámica de Bellman y puede analizar cómo las decisiones de diseño afectan el comportamiento del agente.

Sin embargo, todavía no es suficiente para un sistema operacional. El modelo considera un solo dron, observabilidad perfecta, un entorno estático y una función de recompensa que no incluye varias restricciones del negocio. Por ello, una política óptima en este MDP no sería directamente transferible a la operación real. Además, los resultados de la Tarea 4 muestran que parámetros como (\gamma) y la política inicial pueden cambiar de forma importante el resultado, por lo que el modelo debe validarse antes de usarse en producción.

**Pasos recomendados antes de una implementación completa:**

1. **Calibrar el modelo con datos reales**, como rutas históricas, consumo de batería, tiempos de entrega y tasas de fallo. Esto permitiría ajustar las probabilidades de transición y los valores de recompensa.

2. **Extender el modelo a múltiples drones** de forma gradual, comenzando con pocos agentes y una región reducida, para medir el costo de la coordinación.

3. **Revisar el supuesto de observabilidad completa** y decidir si basta con un estimador de estado o si se necesita una formulación POMDP.

4. **Ampliar el análisis de sensibilidad**, incluyendo no solo (\gamma), sino también los pesos de la función de recompensa. Una mala ponderación puede producir conductas no deseadas, como inmovilidad o uso excesivo de batería.

5. **Definir métricas de negocio y un protocolo de pruebas**, primero en simulación y luego mediante pilotos controlados con hardware real.

6. **Implementar algoritmos de control y RL solo después de estas validaciones**, usando una versión del entorno más realista y calibrada con datos operativos.

En conclusión, el MDP es útil como prototipo académico y metodológico, pero necesita varias extensiones antes de servir como base para un sistema real de logística con drones.
